In [14]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, norm
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.max_columns", None)

# =========================================================
# 0. 데이터 로드 및 필터링 (기존과 동일)
# =========================================================
df = pd.read_csv("C:\DI\[프로젝트2] 데이터 분석 대시보드 제작\Data\전처리 데이터\offer_instance_table.csv")
h1_df = df[(df["offer_type"].isin(["bogo", "discount"])) & (df["is_completed"] == 1)].copy()

# =========================================================
# 1. 프리라이더 정의 재설정
#    - 기존: 미열람 완료(is_viewed==0)만 프리라이더
#    - 변경: 미열람 완료 + "열람은 했지만 완료보다 늦게 열람"한 경우도 포함
#    - 동시각(viewed_time == completed_time)은 일단 "열람이 완료와 같거나 먼저"로 간주(>)
#      더 보수적으로 가려면 >= 로 바꾸면 됨
# =========================================================
기존_프리라이더 = (h1_df["is_viewed"] == 0).astype(int)
새_프리라이더 = ((h1_df["is_viewed"] == 0) | (h1_df["viewed_time"] > h1_df["completed_time"])).astype(int)

print("기존 정의 대비 재분류된 건수:", (새_프리라이더 != 기존_프리라이더).sum())
print("전체 대비 재분류 비율:", (새_프리라이더 != 기존_프리라이더).mean())

h1_df["프리라이더_숫자"] = 새_프리라이더
print("\n재정의 후 전체 프리라이더 비율:", h1_df["프리라이더_숫자"].mean())

기존 정의 대비 재분류된 건수: 4105
전체 대비 재분류 비율: 0.12401438023020453

재정의 후 전체 프리라이더 비율: 0.29709072233467265


In [15]:
# =========================================================
# 1. H1-1: bogo vs discount 프리라이더 비율 비교
# =========================================================
교차표 = pd.crosstab(h1_df["offer_type"], h1_df["프리라이더_숫자"])
교차표.columns = ["열람 후 완료(0)", "미열람 완료=프리라이더(1)"]
print("\n[오퍼타입 x 프리라이더 교차표]")
print(교차표)



# 카이제곱 검정 (연속성 수정 적용/미적용 둘 다 확인)
chi2, p, dof, expected = chi2_contingency(교차표, correction=True)
chi2_nc, p_nc, _, _ = chi2_contingency(교차표, correction=False)

print(f"\n카이제곱(Yates 보정 O) = {chi2:.2f}, p-value = {p:.6f}, 자유도 = {dof}")
print(f"카이제곱(Yates 보정 X) = {chi2_nc:.2f}, p-value = {p_nc:.6f}")

print("\n기대빈도 (모든 셀이 5 이상이어야 카이제곱 근사가 타당함):")
print(pd.DataFrame(expected, index=교차표.index, columns=교차표.columns))

# 효과크기: Cramér's V (2x2이므로 phi coefficient와 동일)
n = 교차표.values.sum()
min_dim = min(교차표.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))
print(f"\nCramér's V = {cramers_v:.4f}  (0.1 미만=미미 / 0.1~0.3=작음 / 0.3~0.5=중간 / 0.5+=큼)")

# 그룹별 프리라이더 비율 + Wilson 신뢰구간
print("\n[오퍼타입별 프리라이더 비율 및 95% 신뢰구간]")
for offer_type in ["bogo", "discount"]:
    sub = h1_df[h1_df["offer_type"] == offer_type]
    count = sub["프리라이더_숫자"].sum()
    nobs = len(sub)
    ci_low, ci_upp = proportion_confint(count, nobs, method="wilson")
    print(f"{offer_type}: 비율 = {count/nobs:.4f}, 95% CI = ({ci_low:.4f}, {ci_upp:.4f}), n = {nobs}")



[오퍼타입 x 프리라이더 교차표]
            열람 후 완료(0)  미열람 완료=프리라이더(1)
offer_type                             
bogo             10941             4560
discount         12326             5274

카이제곱(Yates 보정 O) = 1.16, p-value = 0.281243, 자유도 = 1
카이제곱(Yates 보정 X) = 1.19, p-value = 0.275897

기대빈도 (모든 셀이 5 이상이어야 카이제곱 근사가 타당함):
              열람 후 완료(0)  미열람 완료=프리라이더(1)
offer_type                               
bogo        10895.796713      4605.203287
discount    12371.203287      5228.796713

Cramér's V = 0.0059  (0.1 미만=미미 / 0.1~0.3=작음 / 0.3~0.5=중간 / 0.5+=큼)

[오퍼타입별 프리라이더 비율 및 95% 신뢰구간]
bogo: 비율 = 0.2942, 95% CI = (0.2871, 0.3014), n = 15501
discount: 비율 = 0.2997, 95% CI = (0.2929, 0.3065), n = 17600


In [16]:
# =========================================================
# 2. H1-2: 난이도(difficulty)와 프리라이더 비율의 관계
# =========================================================

# --- 옵션 A-1: 난이도별 교차표 + 카이제곱 (차이 유무만 확인) ---
난이도_교차표 = pd.crosstab(h1_df["difficulty"], h1_df["프리라이더_숫자"])
난이도_교차표.columns = ["열람 후 완료(0)", "미열람 완료=프리라이더(1)"]
print("\n[난이도별 교차표]")
print(난이도_교차표)

chi2_2, p_2, dof_2, expected_2 = chi2_contingency(난이도_교차표)
print(f"\n카이제곱 = {chi2_2:.2f}, p-value = {p_2:.6f}, 자유도 = {dof_2}")
print("기대빈도 5 미만 셀 확인:")
print(pd.DataFrame(expected_2, index=난이도_교차표.index, columns=난이도_교차표.columns))

# --- 옵션 A-2: Cochran-Armitage 추세검정 (방향성까지 확인, scipy/statsmodels에 기본 내장 없어 직접 구현) ---
def cochran_armitage_trend_test(counts_df):
    """
    counts_df: index=순서형 그�지(여기선 difficulty 값), 
               열 순서는 [실패(0), 성공(1)] 형태여야 함 (여기선 [열람후완료, 프리라이더])
    반환: Z 통계량, p-value (양측)
    """
    t = counts_df.index.values.astype(float)   # 그룹별 점수 = difficulty 값 그대로 사용
    n_i = counts_df.sum(axis=1).values
    r_i = counts_df.iloc[:, 1].values          # "프리라이더(1)" 열의 건수

    N, R = n_i.sum(), r_i.sum()
    numerator = np.sum(t * r_i) - (R / N) * np.sum(n_i * t)
    variance = (R / N) * (1 - R / N) * (np.sum(n_i * t**2) - (np.sum(n_i * t))**2 / N)
    z = numerator / np.sqrt(variance)
    p_value = 2 * (1 - norm.cdf(abs(z)))
    return z, p_value

z_stat, p_trend = cochran_armitage_trend_test(난이도_교차표)
print(f"\n[Cochran-Armitage 추세검정] Z = {z_stat:.4f}, p-value = {p_trend:.6f}")
print("Z > 0 이면 난이도가 높아질수록 프리라이더 비율 증가 방향, Z < 0 이면 감소 방향")

# --- 옵션 B: 로지스틱 회귀 (offer_type을 통제하고 난이도의 순수 효과 확인) ---
X = h1_df[["difficulty"]].copy()
X["offer_type_discount"] = (h1_df["offer_type"] == "discount").astype(int)  # bogo=0 기준
X = sm.add_constant(X)
y = h1_df["프리라이더_숫자"]

logit_model = sm.Logit(y, X).fit()
print("\n[로지스틱 회귀: 프리라이더_숫자 ~ difficulty + offer_type(discount 더미)]")
print(logit_model.summary())

print("\n오즈비(odds ratio):")
print(np.exp(logit_model.params))
# difficulty의 오즈비가 1보다 작으면 "난이도가 높을수록 프리라이더 오즈가 낮다" = "쉬울수록 프리라이더 많다"는 H1-2와 방향 일치


[난이도별 교차표]
            열람 후 완료(0)  미열람 완료=프리라이더(1)
difficulty                             
5                 5620             2945
7                 4348              754
10               11999             4122
20                1300             2013

카이제곱 = 2296.26, p-value = 0.000000, 자유도 = 3
기대빈도 5 미만 셀 확인:
              열람 후 완료(0)  미열람 완료=프리라이더(1)
difficulty                               
5            6020.417963      2544.582037
7            3586.243135      1515.756865
10          11331.600465      4789.399535
20           2328.738437       984.261563

[Cochran-Armitage 추세검정] Z = 30.5725, p-value = 0.000000
Z > 0 이면 난이도가 높아질수록 프리라이더 비율 증가 방향, Z < 0 이면 감소 방향
Optimization terminated successfully.
         Current function value: 0.591920
         Iterations 5

[로지스틱 회귀: 프리라이더_숫자 ~ difficulty + offer_type(discount 더미)]
                           Logit Regression Results                           
Dep. Variable:               프리라이더_숫자   No. Observations:                33101
Model: 

In [17]:
# =========================================================
# 3. H1-1 재검증: 난이도(difficulty)를 고정하고 오퍼타입 효과만 재확인
#    - H1-2에서 난이도와 오퍼타입이 강하게 얽혀있는 걸 확인했으므로(심슨의 역설 가능성)
#    - bogo와 discount가 동시에 존재하는 유일한 구간(difficulty==10)만 필터링해서
#      "난이도를 고정한 상태"에서 오퍼타입만의 순수 효과를 재검증
# =========================================================
d10_df = h1_df[h1_df["difficulty"] == 10].copy()

print("난이도=10 인스턴스 수:", len(d10_df))
print(d10_df["offer_type"].value_counts())

교차표_d10 = pd.crosstab(d10_df["offer_type"], d10_df["프리라이더_숫자"])
교차표_d10.columns = ["열람 후 완료(0)", "미열람 완료=프리라이더(1)"]
print("\n[난이도=10 한정 오퍼타입 x 프리라이더 교차표]")
print(교차표_d10)

chi2_d10, p_d10, dof_d10, expected_d10 = chi2_contingency(교차표_d10, correction=True)
print(f"\n카이제곱 = {chi2_d10:.2f}, p-value = {p_d10:.6f}, 자유도 = {dof_d10}")
print("기대빈도:")
print(pd.DataFrame(expected_d10, index=교차표_d10.index, columns=교차표_d10.columns))

n_d10 = 교차표_d10.values.sum()
cramers_v_d10 = np.sqrt(chi2_d10 / (n_d10 * (min(교차표_d10.shape) - 1)))
print(f"\nCramér's V = {cramers_v_d10:.4f}")

print("\n[난이도=10 한정 오퍼타입별 프리라이더 비율 및 95% 신뢰구간]")
for offer_type in ["bogo", "discount"]:
    sub = d10_df[d10_df["offer_type"] == offer_type]
    count = sub["프리라이더_숫자"].sum()
    nobs = len(sub)
    ci_low, ci_upp = proportion_confint(count, nobs, method="wilson")
    print(f"{offer_type}: 비율 = {count/nobs:.4f}, 95% CI = ({ci_low:.4f}, {ci_upp:.4f}), n = {nobs}")

난이도=10 인스턴스 수: 16121
offer_type
discount    9185
bogo        6936
Name: count, dtype: int64

[난이도=10 한정 오퍼타입 x 프리라이더 교차표]
            열람 후 완료(0)  미열람 완료=프리라이더(1)
offer_type                             
bogo              5321             1615
discount          6678             2507

카이제곱 = 33.18, p-value = 0.000000, 자유도 = 1
기대빈도:
             열람 후 완료(0)  미열람 완료=프리라이더(1)
offer_type                              
bogo        5162.524905      1773.475095
discount    6836.475095      2348.524905

Cramér's V = 0.0454

[난이도=10 한정 오퍼타입별 프리라이더 비율 및 95% 신뢰구간]
bogo: 비율 = 0.2328, 95% CI = (0.2230, 0.2429), n = 6936
discount: 비율 = 0.2729, 95% CI = (0.2639, 0.2821), n = 9185


In [18]:
# =========================================================
# H1-3 사전 탐색: 오퍼(offer_id)별 채널 구성 x 난이도 x 프리라이더 비율
#    - 채널이 오퍼마다 고정된 속성이라, 채널 차이가 난이도/오퍼타입과
#      얼마나 겹치는지 먼저 확인해야 "채널만의 순수 효과"인지 판단 가능
# =========================================================
오퍼별_요약 = h1_df.groupby("offer_id").agg(
    오퍼타입=("offer_type", "first"),
    난이도=("difficulty", "first"),
    웹채널=("is_web", "first"),
    이메일채널=("is_email", "first"),
    모바일채널=("is_mobile", "first"),
    소셜채널=("is_social", "first"),
    인스턴스수=("instance_id", "count"),
    프리라이더_비율=("프리라이더_숫자", "mean"),
).sort_values("프리라이더_비율", ascending=False)

print("[오퍼별 채널 구성 x 난이도 x 프리라이더 비율]")
print(오퍼별_요약)

[오퍼별 채널 구성 x 난이도 x 프리라이더 비율]
                                      오퍼타입  난이도  웹채널  이메일채널  모바일채널  소셜채널  \
offer_id                                                                   
0b1e1539f2cc45b7b9fa7c272da2e1d7  discount   20    1      1      0     0   
9b98b8c7a33c4b65b9aebfe6a799e6d9      bogo    5    1      1      1     0   
2906b810c7d4411798c6938adc9daaa5  discount   10    1      1      1     0   
ae264e3637204a6fb9bb56bc8210ddfd      bogo   10    0      1      1     1   
f19421c1d4aa40978ebb69ca19b0e20d      bogo    5    1      1      1     1   
4d5c57ea9a6940dd891ad53e9dbe8da0      bogo   10    1      1      1     1   
2298d6c36e964ae4a3e7e9706d1fb8c2  discount    7    1      1      1     1   
fafdcd668e3743c1bb461111dcafc2a4  discount   10    1      1      1     1   

                                  인스턴스수  프리라이더_비율  
offer_id                                           
0b1e1539f2cc45b7b9fa7c272da2e1d7   3313  0.607606  
9b98b8c7a33c4b65b9aebfe6a799e6d9   4303  0.510574  
29

In [19]:
# =========================================================
# H1-3: 오퍼타입·난이도를 고정한 "매칭 짝" 안에서 소셜 채널 효과 검증
#    - 짝1: bogo, 난이도5 (9b98b8c7 vs f19421c1)
#    - 짝2: discount, 난이도10 (2906b810 vs fafdcd668)
# =========================================================
matched_pairs = {
    "짝1_bogo_난이도5": ["9b98b8c7a33c4b65b9aebfe6a799e6d9", "f19421c1d4aa40978ebb69ca19b0e20d"],
    "짝2_discount_난이도10": ["2906b810c7d4411798c6938adc9daaa5", "fafdcd668e3743c1bb461111dcafc2a4"],
}

for name, offer_ids in matched_pairs.items():
    sub = h1_df[h1_df["offer_id"].isin(offer_ids)].copy()
    교차표 = pd.crosstab(sub["is_social"], sub["프리라이더_숫자"])
    교차표.columns = ["열람 후 완료(0)", "미열람 완료=프리라이더(1)"]
    chi2, p, dof, expected = chi2_contingency(교차표)
    n = 교차표.values.sum()
    cramers_v = np.sqrt(chi2 / (n * (min(교차표.shape) - 1)))

    print(f"\n=== {name} ===")
    print(교차표)
    print(f"카이제곱 = {chi2:.2f}, p-value = {p:.6f}, Cramér's V = {cramers_v:.4f}")


=== 짝1_bogo_난이도5 ===
           열람 후 완료(0)  미열람 완료=프리라이더(1)
is_social                             
0                2106             2197
1                3514              748
카이제곱 = 1064.03, p-value = 0.000000, Cramér's V = 0.3525

=== 짝2_discount_난이도10 ===
           열람 후 완료(0)  미열람 완료=프리라이더(1)
is_social                             
0                2102             1853
1                4576              654
카이제곱 = 1337.06, p-value = 0.000000, Cramér's V = 0.3815


In [20]:
# =========================================================
# H1-3 보충: web 채널 효과 검증 (social·오퍼타입·난이도는 고정)
#    - ae264e36...(bogo, 난이도10, web=0, social=1) vs 4d5c57ea...(bogo, 난이도10, web=1, social=1)
# =========================================================
web_pair = ["ae264e3637204a6fb9bb56bc8210ddfd", "4d5c57ea9a6940dd891ad53e9dbe8da0"]
sub = h1_df[h1_df["offer_id"].isin(web_pair)].copy()

교차표_web = pd.crosstab(sub["is_web"], sub["프리라이더_숫자"])
교차표_web.columns = ["열람 후 완료(0)", "미열람 완료=프리라이더(1)"]
print("[web 채널 x 프리라이더 교차표 (bogo, 난이도10, social=1 고정)]")
print(교차표_web)

chi2_web, p_web, dof_web, expected_web = chi2_contingency(교차표_web)
n_web = 교차표_web.values.sum()
cramers_v_web = np.sqrt(chi2_web / (n_web * (min(교차표_web.shape) - 1)))
print(f"\n카이제곱 = {chi2_web:.2f}, p-value = {p_web:.6f}, Cramér's V = {cramers_v_web:.4f}")

print("\n[web 채널별 프리라이더 비율 및 95% 신뢰구간]")
for web_val in [0, 1]:
    s = sub[sub["is_web"] == web_val]
    count = s["프리라이더_숫자"].sum()
    nobs = len(s)
    ci_low, ci_upp = proportion_confint(count, nobs, method="wilson")
    print(f"web={web_val}: 비율 = {count/nobs:.4f}, 95% CI = ({ci_low:.4f}, {ci_upp:.4f}), n = {nobs}")

[web 채널 x 프리라이더 교차표 (bogo, 난이도10, social=1 고정)]
        열람 후 완료(0)  미열람 완료=프리라이더(1)
is_web                             
0             2582             1053
1             2739              562

카이제곱 = 137.48, p-value = 0.000000, Cramér's V = 0.1408

[web 채널별 프리라이더 비율 및 95% 신뢰구간]
web=0: 비율 = 0.2897, 95% CI = (0.2752, 0.3046), n = 3635
web=1: 비율 = 0.1703, 95% CI = (0.1578, 0.1835), n = 3301
